In [13]:
!pip install pycuda -q

In [14]:
import numpy as np
import struct
import time
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

In [15]:
# ---- Параметры ----
WIDTH      = 1024          # 800x600 ... 1920x1080
HEIGHT     = 768
NUM_SPHERES = 8            # 5..10
NUM_LIGHTS  = 2            # 1..2
MAX_DEPTH   = 5
OUTPUT_FILE = 'output.bmp'
SEED        = 1234

In [16]:
CUDA_SRC = r"""
#include <math.h>
#include <curand_kernel.h>

#define MAX_DEPTH 5
#define EPS       1e-4f


struct Vec3 {
    float x, y, z;
    __device__ Vec3 operator+(const Vec3& v) const { return {x+v.x, y+v.y, z+v.z}; }
    __device__ Vec3 operator-(const Vec3& v) const { return {x-v.x, y-v.y, z-v.z}; }
    __device__ Vec3 operator*(float s)        const { return {x*s,   y*s,   z*s  }; }
    __device__ Vec3 operator*(const Vec3& v)  const { return {x*v.x, y*v.y, z*v.z}; }
    __device__ float dot(const Vec3& v)       const { return x*v.x + y*v.y + z*v.z; }
    __device__ Vec3 normalize() const {
        float len = sqrtf(dot(*this));
        return (len > 0.0f) ? *this * (1.0f/len) : Vec3{0,0,0};
    }
};

__device__ Vec3 reflect(const Vec3& I, const Vec3& N) {
    return I - N * (2.0f * I.dot(N));
}


#define SPH_STRIDE 8
#define LGT_STRIDE 6
#define PLN_STRIDE 9

__device__ bool intersectSphere(const Vec3& ro, const Vec3& rd,
                                const float* sph, float& t) {
    Vec3 c = {sph[0], sph[1], sph[2]};
    float r = sph[3];
    Vec3 oc = ro - c;
    float b = oc.dot(rd);
    float c2 = oc.dot(oc) - r*r;
    float disc = b*b - c2;
    if (disc < 0.0f) return false;
    float sq = sqrtf(disc);
    float t0 = -b - sq;
    float t1 = -b + sq;
    t = (t0 > EPS) ? t0 : t1;
    return t > EPS;
}

__device__ bool intersectPlane(const Vec3& ro, const Vec3& rd,
                               const float* pln, float& t) {
    Vec3 p = {pln[0], pln[1], pln[2]};
    Vec3 n = {pln[3], pln[4], pln[5]};
    float denom = n.dot(rd);
    if (fabsf(denom) < 1e-6f) return false;
    t = (p - ro).dot(n) / denom;
    return t > EPS;
}

// Шахматный пол: чередуются цвет плоскости и тёмный
__device__ Vec3 planeColor(const Vec3& hit, const float* pln) {
    Vec3 base = {pln[6], pln[7], pln[8]};
    int cx = (int)floorf(hit.x * 0.5f);
    int cz = (int)floorf(hit.z * 0.5f);
    bool dark = ((cx + cz) & 1) == 0;
    return dark ? base * 0.35f : base;
}


__device__ Vec3 traceRay(Vec3 ro, Vec3 rd,
                         const float* spheres, int numSpheres,
                         const float* lights,  int numLights,
                         const float* plane) {
    Vec3 finalColor = {0.0f, 0.0f, 0.0f};
    Vec3 atten      = {1.0f, 1.0f, 1.0f};

    for (int depth = 0; depth <= MAX_DEPTH; ++depth) {
        // 1) ищем ближайшее пересечение
        float closestT = 1e20f;
        int   hitSphere = -1;
        bool  hitPlane  = false;

        for (int i = 0; i < numSpheres; ++i) {
            float t;
            if (intersectSphere(ro, rd, spheres + i*SPH_STRIDE, t) && t < closestT) {
                closestT = t;
                hitSphere = i;
                hitPlane  = false;
            }
        }
        float tp;
        if (intersectPlane(ro, rd, plane, tp) && tp < closestT) {
            closestT = tp;
            hitSphere = -1;
            hitPlane  = true;
        }

        if (hitSphere == -1 && !hitPlane) {

            float t = 0.5f * (rd.y + 1.0f);
            Vec3 sky = Vec3{1.0f,1.0f,1.0f} * (1.0f - t)
                     + Vec3{0.5f,0.7f,1.0f} * t;
            finalColor = finalColor + atten * sky * 0.3f;
            break;
        }


        Vec3 hit = ro + rd * closestT;
        Vec3 normal;
        Vec3 surfColor;
        float reflectivity;
        if (hitPlane) {
            normal = Vec3{plane[3], plane[4], plane[5]}.normalize();
            surfColor = planeColor(hit, plane);
            reflectivity = 0.25f;
        } else {
            const float* sph = spheres + hitSphere * SPH_STRIDE;
            normal = (hit - Vec3{sph[0], sph[1], sph[2]}).normalize();
            surfColor = {sph[4], sph[5], sph[6]};
            reflectivity = sph[7];
        }


        Vec3 local = surfColor * 0.1f;


        for (int i = 0; i < numLights; ++i) {
            const float* L = lights + i*LGT_STRIDE;
            Vec3 lightPos  = {L[0], L[1], L[2]};
            Vec3 intensity = {L[3], L[4], L[5]};
            Vec3 toLight = lightPos - hit;
            float lightDist = sqrtf(toLight.dot(toLight));
            Vec3 lightDir = toLight * (1.0f / lightDist);


            Vec3 shadowOrig = hit + normal * EPS;
            bool inShadow = false;
            for (int j = 0; j < numSpheres && !inShadow; ++j) {
                float ts;
                if (intersectSphere(shadowOrig, lightDir,
                                    spheres + j*SPH_STRIDE, ts)
                    && ts < lightDist) {
                    inShadow = true;
                }
            }
            if (inShadow) continue;


            float diff = fmaxf(0.0f, normal.dot(lightDir));
            local = local + surfColor * intensity * diff;

            Vec3 viewDir = rd * -1.0f;
            Vec3 halfV = (lightDir + viewDir).normalize();
            float spec = powf(fmaxf(0.0f, normal.dot(halfV)), 32.0f);
            local = local + intensity * spec * 0.5f;
        }

        finalColor = finalColor + atten * local;

        // 5) рекурсивный шаг — отражённый луч
        if (reflectivity <= 0.0f) break;
        atten = atten * reflectivity;
        if (atten.x + atten.y + atten.z < 0.01f) break;
        ro = hit + normal * EPS;
        rd = reflect(rd, normal).normalize();
    }
    return finalColor;
}


extern "C" __global__
void initScene(float* spheres, int numSpheres,
               float* lights,  int numLights,
               unsigned long seed) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    curandState st;
    curand_init(seed, idx, 0, &st);

    if (idx < numSpheres) {
        float* s = spheres + idx*SPH_STRIDE;

        s[0] = (curand_uniform(&st) * 2.0f - 1.0f) * 4.0f;          // x in [-4,4]
        s[1] = (curand_uniform(&st) * 2.0f - 1.0f) * 1.5f - 0.3f;   // y in [-1.8,1.2]
        s[2] = -3.0f - curand_uniform(&st) * 7.0f;                  // z in [-10,-3]
        s[3] = 0.5f + curand_uniform(&st) * 0.9f;                   // r in [0.5,1.4]
        s[4] = 0.2f + curand_uniform(&st) * 0.8f;                   // R
        s[5] = 0.2f + curand_uniform(&st) * 0.8f;                   // G
        s[6] = 0.2f + curand_uniform(&st) * 0.8f;                   // B
        s[7] = curand_uniform(&st) * 0.6f;                          // reflectivity in [0,0.6]
    }
    if (idx < numLights) {
        float* l = lights + idx*LGT_STRIDE;
        l[0] = (curand_uniform(&st) * 2.0f - 1.0f) * 6.0f;          // x
        l[1] = 4.0f + curand_uniform(&st) * 4.0f;                   // y in [4,8]
        l[2] = -2.0f - curand_uniform(&st) * 4.0f;                  // z

        l[3] = 0.6f + curand_uniform(&st) * 0.4f;
        l[4] = 0.6f + curand_uniform(&st) * 0.4f;
        l[5] = 0.6f + curand_uniform(&st) * 0.4f;
    }
}


extern "C" __global__
void renderKernel(unsigned char* image, int width, int height,
                  const float* spheres, int numSpheres,
                  const float* lights,  int numLights,
                  const float* plane) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= width || y >= height) return;

    float aspect = (float)width / (float)height;
    float fovScale = tanf(60.0f * 3.14159265f / 360.0f);   // FOV = 60°

    Vec3 ro = {0.0f, 0.0f, 0.0f};
    Vec3 rd = {
        (2.0f * (x + 0.5f) / width  - 1.0f) * aspect * fovScale,
        (1.0f - 2.0f * (y + 0.5f) / height)        * fovScale,
        -1.0f
    };
    rd = rd.normalize();

    Vec3 c = traceRay(ro, rd, spheres, numSpheres, lights, numLights, plane);


    c.x = sqrtf(fminf(1.0f, fmaxf(0.0f, c.x)));
    c.y = sqrtf(fminf(1.0f, fmaxf(0.0f, c.y)));
    c.z = sqrtf(fminf(1.0f, fmaxf(0.0f, c.z)));


    int p = (y * width + x) * 3;
    image[p + 0] = (unsigned char)(c.x * 255.0f);   // R
    image[p + 1] = (unsigned char)(c.y * 255.0f);   // G
    image[p + 2] = (unsigned char)(c.z * 255.0f);   // B
}
"""


mod = SourceModule(CUDA_SRC, no_extern_c=True, options=['-std=c++14'])
init_scene    = mod.get_function('initScene')
render_kernel = mod.get_function('renderKernel')

In [17]:
def save_bmp(filename, image_rgb, width, height):

    img = image_rgb.reshape(height, width, 3)

    bgr = img[::-1, :, ::-1]

    row_bytes = width * 3
    pad = (4 - row_bytes % 4) % 4
    if pad:
        padded = np.zeros((height, row_bytes + pad), dtype=np.uint8)
        padded[:, :row_bytes] = bgr.reshape(height, row_bytes)
        pixel_data = padded.tobytes()
    else:
        pixel_data = bgr.tobytes()

    file_size = 54 + len(pixel_data)
    header = struct.pack('<2sIHHI', b'BM', file_size, 0, 0, 54)
    dib    = struct.pack('<IiiHHIIiiII',
                         40, width, height, 1, 24, 0,
                         len(pixel_data), 2835, 2835, 0, 0)
    with open(filename, 'wb') as f:
        f.write(header + dib + pixel_data)
    print(f'Saved: {filename}  ({width}x{height})')

In [18]:

spheres_host = np.zeros(NUM_SPHERES * 8, dtype=np.float32)
lights_host  = np.zeros(NUM_LIGHTS  * 6, dtype=np.float32)

spheres_gpu = cuda.mem_alloc(spheres_host.nbytes)
lights_gpu  = cuda.mem_alloc(lights_host.nbytes)
cuda.memcpy_htod(spheres_gpu, spheres_host)
cuda.memcpy_htod(lights_gpu,  lights_host)

n = max(NUM_SPHERES, NUM_LIGHTS)
init_scene(spheres_gpu, np.int32(NUM_SPHERES),
           lights_gpu,  np.int32(NUM_LIGHTS),
           np.uint64(SEED),
           block=(max(n, 1), 1, 1), grid=(1, 1))
cuda.Context.synchronize()

cuda.memcpy_dtoh(spheres_host, spheres_gpu)
cuda.memcpy_dtoh(lights_host,  lights_gpu)


plane_host = np.array([0.0, -2.0, 0.0,
                       0.0,  1.0, 0.0,
                       0.8,  0.8, 0.8], dtype=np.float32)
plane_gpu  = cuda.mem_alloc(plane_host.nbytes)
cuda.memcpy_htod(plane_gpu, plane_host)

# Печатаем сцену
print('Сферы:')
for i in range(NUM_SPHERES):
    s = spheres_host[i*8:(i+1)*8]
    print(f'  #{i}: center=({s[0]:+.2f},{s[1]:+.2f},{s[2]:+.2f}) '
          f'r={s[3]:.2f}  color=({s[4]:.2f},{s[5]:.2f},{s[6]:.2f}) '
          f'refl={s[7]:.2f}')
print('Источники света:')
for i in range(NUM_LIGHTS):
    l = lights_host[i*6:(i+1)*6]
    print(f'  #{i}: pos=({l[0]:+.2f},{l[1]:+.2f},{l[2]:+.2f}) '
          f'intensity=({l[3]:.2f},{l[4]:.2f},{l[5]:.2f})')

Сферы:
  #0: center=(-2.84,-0.50,-9.09) r=0.91  color=(0.89,0.67,0.36) refl=0.47
  #1: center=(+2.56,+0.98,-6.58) r=0.98  color=(0.85,0.21,0.56) refl=0.48
  #2: center=(+0.40,+0.64,-8.48) r=1.17  color=(0.63,0.36,0.20) refl=0.06
  #3: center=(-1.64,-0.87,-7.34) r=0.98  color=(0.60,0.92,0.23) refl=0.60
  #4: center=(+3.32,-0.13,-6.88) r=0.55  color=(0.72,0.77,0.41) refl=0.57
  #5: center=(+2.95,-0.30,-4.50) r=0.84  color=(0.45,0.70,0.75) refl=0.47
  #6: center=(-1.42,-1.18,-3.83) r=0.62  color=(0.34,0.54,0.77) refl=0.53
  #7: center=(+2.26,-1.43,-9.96) r=0.57  color=(0.76,0.61,0.37) refl=0.27
Источники света:
  #0: pos=(-2.39,+5.47,-4.31) intensity=(0.88,0.80,0.89)
  #1: pos=(-4.36,+5.51,-2.41) intensity=(0.90,0.82,0.89)


In [19]:

image = np.zeros(WIDTH * HEIGHT * 3, dtype=np.uint8)
image_gpu = cuda.mem_alloc(image.nbytes)

block = (16, 16, 1)
grid  = ((WIDTH  + block[0] - 1) // block[0],
         (HEIGHT + block[1] - 1) // block[1])

start_evt = cuda.Event()
end_evt   = cuda.Event()

start_evt.record()
render_kernel(image_gpu,
              np.int32(WIDTH), np.int32(HEIGHT),
              spheres_gpu, np.int32(NUM_SPHERES),
              lights_gpu,  np.int32(NUM_LIGHTS),
              plane_gpu,
              block=block, grid=grid)
end_evt.record()
end_evt.synchronize()

gpu_ms = start_evt.time_till(end_evt)
print(f'Время рендеринга на GPU: {gpu_ms:.2f} мс')

cuda.memcpy_dtoh(image, image_gpu)
save_bmp(OUTPUT_FILE, image, WIDTH, HEIGHT)

Время рендеринга на GPU: 0.79 мс
Saved: output.bmp  (1024x768)


In [20]:
spheres_gpu.free()
lights_gpu.free()
plane_gpu.free()
image_gpu.free()